# 03 RL Model Training (DDQN)
Training the agent to classify market states for VaR adjustment.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from src.feature_engineering import create_feature_matrix, normalize_features
from src.ddqn_agent import DDQNAgent
from src.classification_models import train_baseline_models

log_returns = pd.read_csv('../data/processed/log_returns.csv', index_col=0, parse_dates=True)
adj_close = pd.read_csv('../data/processed/adj_close.csv', index_col=0, parse_dates=True)
risk_data = pd.read_csv('../data/processed/risk_data.csv', index_col=0, parse_dates=True)

# 1. Feature Engineering
full_features = create_feature_matrix(log_returns, adj_close)
# Merge with risk labels and volatility from Step 2
dataset = full_features.join(risk_data[['volatility', 'risk_label']], how='inner')

X = dataset.drop(columns=['risk_label'])
y = dataset['risk_label']

X_scaled, scaler = normalize_features(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, shuffle=False)

print(f"X shape: {X_scaled.shape}")

In [ ]:
print("Training Baselines...")
baselines, trained_models = train_baseline_models(X_train, y_train, X_test, y_test)
baseline_df = pd.DataFrame(baselines).T
print(baseline_df)
baseline_df.to_csv('../results/metrics_baselines.csv')

In [ ]:
minority_ratio = y_train.mean()
agent = DDQNAgent(state_size=X_train.shape[1], action_size=2, minority_ratio=minority_ratio)
batch_size = 32
EPISODES = 10 

rewards = []
for e in range(EPISODES):
    total_reward = 0
    for i in range(len(X_train)):
        state = X_train.iloc[i].values.reshape(1, -1)
        action = agent.act(state)
        reward = agent.calculate_reward(action, y_train.iloc[i])
        
        next_state = X_train.iloc[min(i+1, len(X_train)-1)].values.reshape(1, -1)
        done = (i == len(X_train) - 1)
        
        agent.remember(state, action, reward, next_state, done)
        total_reward += reward
        
        if len(agent.memory) > batch_size:
            # Only train every few steps to speed up
            if i % 10 == 0: agent.replay(batch_size)
            
    agent.update_target_model()
    rewards.append(total_reward)
    print(f"Episode: {e+1}/{EPISODES}, Total Reward: {total_reward:.2f}")

plt.plot(rewards)
plt.title('RL Training Reward Curve')
plt.savefig('../results/plots/rl_reward_curve.png')
plt.show()

In [ ]:
# Agent Prediction on Test Set
y_rl_pred = []
for i in range(len(X_test)):
    state = X_test.iloc[i].values.reshape(1, -1)
    action = agent.act(state)
    y_rl_pred.append(action)

test_results = risk_data.loc[X_test.index].copy()
test_results['rl_prediction'] = y_rl_pred
test_results.to_csv('../results/rl_test_predictions.csv')
print("Saved RL predictions for backtesting.")